# 0345 · Auditorías pedidas por el director (evaluación del 04-11-2026)

**Qué hace.** Solo cuenta y resume; no cambia la base ni los modelos. Responde las preguntas que hay que contestar **antes** de reconstruir la base:

| Sección | Ajuste | Pregunta |
| --- | --- | --- |
| 1 | C1 | ¿Qué estados de envío existen y cuántas guías hay de cada uno por mes en los archivos crudos de 0100? ¿Cuántas quedaron fuera por el filtro `ID_ESTADOENVIO >= 3`? |
| 2 | C1 | ¿Cuándo se extrajeron los datos? ¿Qué fecha de corte corresponde? |
| 3 | A2 | ¿Cuántas guías salen a reparto o se entregan antes de su admisión, y a qué distancia? ¿Cuántas cambian de T0 si se exige una salida posterior a la admisión? |
| 4 | A3 | ¿Las guías «digitalizadas» tienen un perfil de entrega distinto? |
| 5 | D2 | ¿En qué tramo se agota el plazo: de la admisión a la bodega o de la bodega a la salida? ¿Cambia según el origen? |

**Entradas.** Las mismas que ya están en el Volume: `LOG_GUIAS/LOG_GUIAS*.parquet` (crudos de 0100), `LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet` (0300), los movimientos, la base v3 (`V3/GOLD_V3_ENVIO.parquet`) y la sal de la guía.

**Salidas.** Solo CSV agregados `P10_*.csv` en `REVISION_2026_11/V4/AUDITORIA/`. Ninguna celda muestra guías, clientes ni nombres.

In [ ]:
%pip install "duckdb<2"

In [ ]:
# Rutas (las mismas de 0300 y 0330)
import os, json, datetime
import duckdb, pandas as pd
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
RUTA_GUIAS_CRUDAS = f"{PATH_DATABRICKS}LOG_GUIAS/LOG_GUIAS*.parquet"              # 0100, sin filtro de estado
RUTA_GUIAS_0300   = f"{PATH_DATABRICKS}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"    # 0300, con ID_ESTADOENVIO >= 3
RUTA_MOVS_GLOB    = f"{PATH_DATABRICKS}LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO*.parquet"
RUTA_GOLD_V3      = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/GOLD_V3_ENVIO.parquet"
PAR = f"{PATH_DATABRICKS}PARAMETRICAS/"
RUTA_EST_ENVIO, RUTA_EST_GUIA = f"{PAR}PAR.ESTADOSENVIO.parquet", f"{PAR}PAR.ESTADOSGUIA.parquet"
RUTA_SAL_GUIA = f"{PATH_DATABRICKS}SEGURIDAD/sal_guia.txt"      # la misma de 0330; no se imprime
CARPETA = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/AUDITORIA/"
os.makedirs(CARPETA, exist_ok=True)
ID_SALIDA_ZONA, ID_BOGOTA = 9, 10
SALIDAS = {}
def guardar(nombre, t):
    SALIDAS[nombre] = t
    t.to_csv(f"{CARPETA}P10_{nombre}.csv", index=False)
    display(t)
def columnas(ruta):
    return set(duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{ruta}')").df().column_name)

In [ ]:
# GUARDA DE LA RESERVA (C2): septiembre y octubre no se leen hasta registrar P4b_decision_v2.json.
# Si en las carpetas de desarrollo hay archivos de 202609 o posteriores, el cuaderno se detiene sin abrirlos.
import glob as _glob, re as _re, os as _os
_reserva = []
for _carpeta in ["LOG_GUIAS", "LOG_SEGUIMIENTO_ENVIO", "LOG_DOCUMENTOS"]:
    for _f in _glob.glob(f"{PATH_DATABRICKS}{_carpeta}/*.parquet"):
        _m = _re.search(r"(20\d{4})\D*\.parquet$", _os.path.basename(_f))
        if _m and int(_m.group(1)) >= 202609:
            _reserva.append(f"{_carpeta}/{_os.path.basename(_f)}")
assert not _reserva, ("Hay archivos de la reserva en las carpetas de desarrollo: " + ", ".join(sorted(_reserva)) +
                      ". Moverlos sin abrirlos a RESERVA_POSTERIOR/<misma carpeta>/ (celda del LEEME) y volver a correr.")
print("Guarda de la reserva: no hay archivos de septiembre en adelante en las carpetas de desarrollo.")

## 1. Estados de envío y guías por mes (C1)

El filtro de 0300 (`ID_ESTADOENVIO >= 3`) deja fuera las guías que no se habían cerrado al extraer. Aquí se cuentan **en los crudos**, con los mismos filtros de destino y longitud de guía, pero sin el de estado.

In [ ]:
estados_envio = duckdb.sql(f"SELECT DISTINCT ID_ESTADOENVIO, NOMBRE_ESTADOENVIO FROM read_parquet('{RUTA_EST_ENVIO}') ORDER BY 1").df()
estados_guia  = duckdb.sql(f"SELECT DISTINCT ID_ESTADOGUIA, NOMBRE_ESTADOGUIA FROM read_parquet('{RUTA_EST_GUIA}') ORDER BY 1").df()
guardar("1a_estados_envio", estados_envio)
guardar("1b_estados_guia", estados_guia)

In [ ]:
conteo = duckdb.sql(f"""
    SELECT strftime(G.FECHA_ENVIO, '%Y-%m') AS mes, G.ID_ESTADOENVIO, E.NOMBRE_ESTADOENVIO,
           COUNT(*) AS guias, COUNT(DISTINCT G.NUMERO_GUIA) AS guias_distintas,
           COUNT(G.FECHA_ENTREGA) AS con_fecha_entrega,
           COUNT(*) FILTER (WHERE G.ID_ESTADOENVIO >= 3) AS pasan_filtro_0300
    FROM read_parquet('{RUTA_GUIAS_CRUDAS}') G
    LEFT JOIN (SELECT DISTINCT ID_ESTADOENVIO, NOMBRE_ESTADOENVIO FROM read_parquet('{RUTA_EST_ENVIO}')) E USING (ID_ESTADOENVIO)
    WHERE G.ID_CIUDAD_DESTINO = {ID_BOGOTA} AND length(G.NUMERO_GUIA::VARCHAR) >= 8
    GROUP BY ALL ORDER BY ALL""").df()
guardar("1c_guias_mes_estado", conteo)
print("Guías por mes y estado (crudos):")
display(conteo.pivot_table(index="mes", columns="ID_ESTADOENVIO", values="guias", aggfunc="sum", fill_value=0))
excluidas = conteo[conteo.ID_ESTADOENVIO < 3]
print(f"Guías que el filtro >= 3 dejó fuera: {int(excluidas.guias.sum()):,}; "
      f"guías repetidas en los crudos: {int((conteo.guias - conteo.guias_distintas).sum()):,}")

## 2. Momento de la extracción (C1)

El director encontró que la extracción de guías terminó el 28-08 a las 18:25, pero el código usa como corte el 31-08 a las 23:59. Se revisa la fecha de extracción por mes de admisión, en guías y movimientos, y cuántas admisiones son posteriores. El **corte propuesto** es el menor de los dos máximos.

In [ ]:
ext_mes = duckdb.sql(f"""
    SELECT strftime(FECHA_ENVIO, '%Y-%m') AS mes, COUNT(*) AS guias,
           MIN(FECHA_EXTRACCION_DW) AS extraccion_min, MAX(FECHA_EXTRACCION_DW) AS extraccion_max,
           MAX(FECHA_ENVIO) AS ultima_admision,
           COUNT(*) FILTER (WHERE FECHA_ENVIO > FECHA_EXTRACCION_DW) AS admitidas_despues_de_su_extraccion
    FROM read_parquet('{RUTA_GUIAS_CRUDAS}')
    WHERE ID_CIUDAD_DESTINO = {ID_BOGOTA} AND length(NUMERO_GUIA::VARCHAR) >= 8
    GROUP BY 1 ORDER BY 1""").df()
guardar("2a_extraccion_por_mes", ext_mes)

max_guias = duckdb.sql(f"SELECT MAX(FECHA_EXTRACCION_DW) FROM read_parquet('{RUTA_GUIAS_CRUDAS}')").fetchone()[0]
col_movs = columnas(RUTA_MOVS_GLOB)
campo = "FECHA_EXTRACCION_DW" if "FECHA_EXTRACCION_DW" in col_movs else "FECHA_HORA_MOVIMIENTO"
max_movs = duckdb.sql(f"SELECT MAX({campo}) FROM read_parquet('{RUTA_MOVS_GLOB}')").fetchone()[0]
max_mov_real = duckdb.sql(f"SELECT MAX(FECHA_HORA_MOVIMIENTO) FROM read_parquet('{RUTA_MOVS_GLOB}')").fetchone()[0]
corte = min(pd.Timestamp(max_guias), pd.Timestamp(max_movs))
despues = duckdb.sql(f"""SELECT COUNT(*) FROM read_parquet('{RUTA_GUIAS_CRUDAS}')
                         WHERE ID_CIUDAD_DESTINO = {ID_BOGOTA} AND FECHA_ENVIO > TIMESTAMP '{corte}'""").fetchone()[0]
resumen_corte = pd.DataFrame([{"max_extraccion_guias": max_guias, f"max_{campo.lower()}_movimientos": max_movs,
                               "ultimo_movimiento": max_mov_real, "corte_propuesto": corte,
                               "admisiones_posteriores_al_corte": despues}])
guardar("2b_corte", resumen_corte)

## 3. Salidas y entregas anteriores a la admisión (A2)

Sobre las guías de 0300 y **todos** sus movimientos: primera salida a zona (T0 actual), primera salida **posterior** a la admisión (T0 propuesto) y cuántas guías cambian o se quedan sin T0. Las distancias se agrupan para saber si es un desfase de registro (minutos u horas) o números de guía reutilizados (días o semanas).

In [ ]:
a2 = duckdb.sql(f"""
    WITH G AS (SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, FECHA_ENVIO, FECHA_ENTREGA FROM read_parquet('{RUTA_GUIAS_0300}')),
    S AS (SELECT CAST(M.NUMERO_GUIA AS BIGINT) AS guia,
                 MIN(M.FECHA_HORA_MOVIMIENTO) AS t0_actual,
                 MIN(M.FECHA_HORA_MOVIMIENTO) FILTER (WHERE M.FECHA_HORA_MOVIMIENTO >= G.FECHA_ENVIO) AS t0_propuesto
          FROM read_parquet('{RUTA_MOVS_GLOB}') M JOIN G ON G.guia = CAST(M.NUMERO_GUIA AS BIGINT)
          WHERE M.ID_PROCESO = {ID_SALIDA_ZONA} GROUP BY 1),
    X AS (SELECT G.*, S.t0_actual, S.t0_propuesto,
                 (epoch_ms(S.t0_actual) - epoch_ms(G.FECHA_ENVIO)) / 3600000.0 AS h_t0,
                 G.FECHA_ENTREGA = date_trunc('day', G.FECHA_ENTREGA) AS sin_hora
          FROM G JOIN S USING (guia))
    SELECT strftime(FECHA_ENVIO, '%Y-%m') AS mes, COUNT(*) AS guias_con_salida,
           COUNT(*) FILTER (WHERE h_t0 < 0)                      AS t0_antes_de_admision,
           COUNT(*) FILTER (WHERE h_t0 < 0 AND h_t0 >= -1)       AS t0_antes_hasta_1h,
           COUNT(*) FILTER (WHERE h_t0 < -1 AND h_t0 >= -24)     AS t0_antes_1_a_24h,
           COUNT(*) FILTER (WHERE h_t0 < -24 AND h_t0 >= -168)   AS t0_antes_1_a_7_dias,
           COUNT(*) FILTER (WHERE h_t0 < -168)                   AS t0_antes_mas_de_7_dias,
           COUNT(*) FILTER (WHERE t0_actual < t0_propuesto)      AS cambian_de_t0,
           COUNT(*) FILTER (WHERE t0_propuesto IS NULL)          AS quedan_sin_t0,
           COUNT(*) FILTER (WHERE NOT sin_hora AND FECHA_ENTREGA < FECHA_ENVIO)              AS entrega_antes_de_admision_con_hora,
           COUNT(*) FILTER (WHERE sin_hora AND CAST(FECHA_ENTREGA AS DATE) < CAST(FECHA_ENVIO AS DATE)) AS entrega_dia_anterior_sin_hora,
           COUNT(*) FILTER (WHERE sin_hora AND CAST(FECHA_ENTREGA AS DATE) = CAST(FECHA_ENVIO AS DATE)) AS entrega_mismo_dia_sin_hora
    FROM X GROUP BY 1 ORDER BY 1""").df()
total = a2.drop(columns="mes").sum().to_frame().T.assign(mes="TOTAL")
guardar("3_t0_y_entrega_antes_de_admision", pd.concat([a2, total], ignore_index=True))

## 4. Perfil de entrega por estado de la guía (A3)

Solo guías entregadas de la base v3 (población principal: SLA único o por regla). La guía de la base está seudonimizada; para traer el indicador de receptor de 0300 se calcula el mismo seudónimo con la sal (no se imprime).

In [ ]:
with open(RUTA_SAL_GUIA) as f:
    _sal = f.read().strip()
a3 = duckdb.sql(f"""
    WITH R AS (SELECT md5(CAST(CAST(NUMERO_GUIA AS BIGINT) AS VARCHAR) || '{_sal}') AS guia,
                      (IDEN_PERS_RECIBE IS NOT NULL OR NOMB_PERS_RECIBE IS NOT NULL) AS hay_receptor
               FROM read_parquet('{RUTA_GUIAS_0300}')),
    B AS (SELECT * FROM read_parquet('{RUTA_GOLD_V3}')
          WHERE estado_desenlace = 'ENTREGADO' AND origen_sla IN ('SLA_UNICO', 'SLA_POR_REGLA'))
    SELECT EG.NOMBRE_ESTADOGUIA AS estado_guia, B.estado_envio, COUNT(*) AS guias,
           AVG(R.hay_receptor::INT)                                        AS pct_con_receptor,
           median(B.h_ultima_milla)                                        AS mediana_h_t0_a_entrega,
           quantile_cont(B.h_ultima_milla, 0.9)                            AS p90_h_t0_a_entrega,
           AVG((hour(B.fecha_entrega) BETWEEN 7 AND 20)::INT)              AS pct_entrega_7_a_20h,
           AVG((B.fecha_entrega = date_trunc('day', B.fecha_entrega))::INT) AS pct_sin_hora,
           AVG((B.h_ultima_milla < 0)::INT)                                AS pct_entrega_antes_de_t0,
           AVG(B.incumplimiento_sla)                                       AS incumplimiento,
           AVG(B.vencido_antes_de_t0::INT)                                 AS pct_vencido_antes_de_t0,
           AVG(B.incumplimiento_sla) FILTER (WHERE NOT B.vencido_antes_de_t0) AS incumplimiento_con_margen
    FROM B LEFT JOIN R USING (guia)
    LEFT JOIN (SELECT DISTINCT ID_ESTADOGUIA, NOMBRE_ESTADOGUIA FROM read_parquet('{RUTA_EST_GUIA}')) EG USING (ID_ESTADOGUIA)
    GROUP BY ALL ORDER BY guias DESC""").df()
guardar("4_perfil_por_estado_guia", a3)

## 5. Tramos antes de la salida a reparto, por origen (D2)

`primera_entrada` = primer movimiento de proceso 5 o 6 entre la admisión y T0. Se reportan mediana y p90 de admisión→bodega y bodega→T0, por origen (Bogotá u otra ciudad) y por estado en T0 (vencida o con margen). La segunda tabla da el porcentaje de vencidas antes de T0 por origen, que el director citó de 0340 (49,1 % frente a 43,9 %).

In [ ]:
tramos_sql = f"""
    WITH G AS (SELECT CAST(NUMERO_GUIA AS BIGINT) AS g_real, FECHA_ENVIO,
                      md5(CAST(CAST(NUMERO_GUIA AS BIGINT) AS VARCHAR) || '{_sal}') AS guia
               FROM read_parquet('{RUTA_GUIAS_0300}')),
    B AS (SELECT guia, t0, ID_CIUDAD_ORIGEN, vencido_antes_de_t0, incumplimiento_sla, sla_dias
          FROM read_parquet('{RUTA_GOLD_V3}')
          WHERE origen_sla IN ('SLA_UNICO', 'SLA_POR_REGLA') AND estado_desenlace NOT IN ('DEVUELTO', 'SINIESTRADO')),
    E AS (SELECT G.guia, MIN(M.FECHA_HORA_MOVIMIENTO) AS primera_entrada
          FROM read_parquet('{RUTA_MOVS_GLOB}') M
          JOIN G ON G.g_real = CAST(M.NUMERO_GUIA AS BIGINT)
          JOIN B ON B.guia = G.guia
          WHERE M.ID_PROCESO IN (5, 6) AND M.FECHA_HORA_MOVIMIENTO >= G.FECHA_ENVIO AND M.FECHA_HORA_MOVIMIENTO <= B.t0
          GROUP BY 1),
    X AS (SELECT B.*, CASE WHEN B.ID_CIUDAD_ORIGEN = {ID_BOGOTA} THEN 'BOGOTA' ELSE 'OTRA CIUDAD' END AS origen,
                 (epoch_ms(E.primera_entrada) - epoch_ms(G.FECHA_ENVIO)) / 3600000.0 AS h_admision_a_bodega,
                 (epoch_ms(B.t0) - epoch_ms(E.primera_entrada)) / 3600000.0         AS h_bodega_a_t0,
                 (epoch_ms(B.t0) - epoch_ms(G.FECHA_ENVIO)) / 3600000.0             AS h_admision_a_t0
          FROM B JOIN G USING (guia) LEFT JOIN E USING (guia))
"""
tramos = duckdb.sql(tramos_sql + """
    SELECT origen, vencido_antes_de_t0, COUNT(*) AS guias, AVG((h_admision_a_bodega IS NOT NULL)::INT) AS pct_con_entrada_bodega,
           median(h_admision_a_bodega) AS mediana_h_admision_a_bodega, quantile_cont(h_admision_a_bodega, 0.9) AS p90_h_admision_a_bodega,
           median(h_bodega_a_t0) AS mediana_h_bodega_a_t0, quantile_cont(h_bodega_a_t0, 0.9) AS p90_h_bodega_a_t0,
           median(h_admision_a_t0) AS mediana_h_admision_a_t0, quantile_cont(h_admision_a_t0, 0.9) AS p90_h_admision_a_t0
    FROM X GROUP BY ALL ORDER BY ALL""").df()
guardar("5a_tramos_por_origen", tramos)
origen = duckdb.sql(tramos_sql + """
    SELECT origen, COUNT(*) AS guias, AVG(vencido_antes_de_t0::INT) AS pct_vencido_antes_de_t0,
           AVG(incumplimiento_sla) AS incumplimiento, median(h_admision_a_t0) AS mediana_h_admision_a_t0
    FROM X GROUP BY 1 ORDER BY 1""").df()
guardar("5b_vencidas_por_origen", origen)
del _sal

In [ ]:
# Manifiesto: qué se corrió y qué salió (sin la sal)
with open(f"{CARPETA}P10_manifiesto.json", "w") as f:
    json.dump({"fecha_corrida": datetime.datetime.now().isoformat(), "duckdb": duckdb.__version__,
               "entradas": [RUTA_GUIAS_CRUDAS, RUTA_GUIAS_0300, RUTA_MOVS_GLOB, RUTA_GOLD_V3],
               "salidas": sorted(f"P10_{k}.csv" for k in SALIDAS)}, f, indent=2)
print("Listo:", sorted(SALIDAS))

## Qué devolver

Los archivos `P10_*.csv` y `P10_manifiesto.json` de `REVISION_2026_11/V4/AUDITORIA/`. Con ellos se fijan los parámetros de 0305 (estados que entran) y 0330 v4 (corte y tolerancia de T0).